# Exercise solutions — Transfer learning and fine-tuning

**DeepVision Masterclass · Solutions companion**

These are worked reference solutions for the exercises in the matching numbered lesson. Run the parent lesson first when variables or datasets are reused. Dataset-specific experiments are left as clearly marked tasks; no performance scores are fabricated.

## Exercise 1 — Frozen features vs fine-tuning
Compare a frozen backbone with a model where only the final ResNet stage is trainable. Use the same split and early-stopping criterion.

In [ ]:
from torchvision.models import resnet50,ResNet50_Weights
import torch.nn as nn
weights=ResNet50_Weights.DEFAULT
frozen=resnet50(weights=weights)
for p in frozen.parameters(): p.requires_grad=False
frozen.fc=nn.Linear(frozen.fc.in_features,num_classes)
partial=resnet50(weights=weights)
for p in partial.parameters(): p.requires_grad=False
for p in partial.layer4.parameters(): p.requires_grad=True
partial.fc=nn.Linear(partial.fc.in_features,num_classes)
# Optimizer must include only parameters with requires_grad=True.

## Exercise 2 — EfficientNetV2 / MobileNetV3 alternatives
Use the official weights and preprocessing transforms for each model; don't reuse ResNet normalization blindly.

In [ ]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights, mobilenet_v3_large, MobileNet_V3_Large_Weights
m_eff=efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
eff_preprocess=EfficientNet_V2_S_Weights.DEFAULT.transforms()
m_mobile=mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
mobile_preprocess=MobileNet_V3_Large_Weights.DEFAULT.transforms()
# Replace classifier heads with task-specific output dimensions; compare parameter count, latency and validation macro-F1.

## Exercise 3 — Latency and parameter count
Warm up the model, synchronize CUDA around timed regions, and report batch size and device.

In [ ]:
import time, torch
def benchmark_model(model, sample, warmup=10, runs=50):
    model.eval(); device=next(model.parameters()).device
    with torch.inference_mode():
        for _ in range(warmup): model(sample)
        if device.type=='cuda': torch.cuda.synchronize()
        start=time.perf_counter()
        for _ in range(runs): model(sample)
        if device.type=='cuda': torch.cuda.synchronize()
    params=sum(p.numel() for p in model.parameters())
    return {'parameters':params,'ms_per_batch':1000*(time.perf_counter()-start)/runs,'device':str(device)}
# sample = torch.randn(1,3,224,224,device=device)
# benchmark_model(frozen.to(device),sample)

## Exercise 4 — Controlled comparison
Report split IDs, class mapping, input size, augmentation, pretrained weight identifier, frozen/unfrozen layers, optimizer, seeds, best validation epoch, test macro-F1 and latency.

## Reporting template
Record the dataset/source and license, split strategy, seed(s), preprocessing, model configuration, optimizer, epochs, best validation checkpoint, final test metrics, hardware and runtime. Replace illustrative code comments with measured results from your own run.